## Тест LLM-кандидатів — Google Colab, v8 (MamayLM проти Lapa LLM)

**Чому v8:** звужено до двох україномовних кандидатів того самого розміру й
тієї самої бази (обидва — фʼютюни Gemma-3-12B, обидва — Gemma ToU):
- **MamayLM-Gemma-3-12B-IT-v2.0** (INSAIT, Софія/ETH Zürich) — вже тестований
  раніше, поки що найкращий результат сесії.
- **Lapa LLM v0.1.2-instruct** (lapa-llm — Український католицький
  університет, AGH Краків, КПІ, Львівська політехніка) — новий кандидат,
  ще не тестований у цій сесії. Самозаявлена перевага: адаптація токенізатора
  під українську (заміна 80 000 з 250 000 токенів), що, за твердженням
  розробників, дає ~1.5x менше токенів на українському тексті і, відповідно,
  швидший інференс — **це вендорська заява, незалежно не перевірена**, як і
  свого часу заявка INSAIT про перевагу MamayLM на ЗНО.

Обидві моделі — той самий розмір (12B, 7.3 GB у Q4_K_M) і та сама базова
архітектура (Gemma-3-12B), тож порівняння чесне: різниця у результатах
відображає різницю в підході до українського дотренування, а не в розмірі
чи базі.

**Чому Colab:** локальний ноут (16 GB RAM, CPU-only) впирався в стелю десь між
12B і 24B — Mistral-Small-24B викликав 12-годинне зависання через своп на
диск. Colab (GPU + більше RAM) знімає це обмеження. Обидві моделі тут (7.3 GB
кожна) комфортно влазять у VRAM навіть звичайної T4 (16 GB).

**Що змінилось у самому дизайні задач (3-агентний перегляд, v4):**

1. **Агент-дизайнер** побудував задачі, спираючись на architecture-proposal.md
   і project-expectations.md (не довільно) — включно з фактичною схемою бази
   (вимір+значення для фактів, а не плаский enum).
2. **Агент-перевіряючий** знайшов і ми виправили: два кейси видавали вигадане
   за реальне (один моделював рукописний документ, який архітектура прямо
   виключає з автообробки), самосуперечність у нормалізації відмінка
   прізвища, і суперечність між задачами А і В щодо формату параметра
   підрозділу (алфавітно-числові коди частин "А0000" проти передбачення
   "просто число"). Усі тестові документи нижче тепер або дослівні цитати
   реального OCR наших власних шаблонів (`docs/відпустка/`, вже
   розпізнаних у попередніх ноутбуках цієї сесії), або чесно позначені як
   синтетичні — без змішування цих двох категорій під одним "джерелом".
3. **Агент-промптолог** переписав промпти за джерелами 2025-2026
   (Anthropic, JSONSchemaBench arXiv:2501.10868, "When Correct Isn't Usable"
   arXiv:2605.02363, "Schema Key Wording as an Instruction Channel"
   arXiv:2604.14862, "Lost in the Middle at Birth" arXiv:2603.10123):
   формат гарантує JSON-схема (grammar-constrained decoding), тому текстові
   "поверни лише JSON" інструкції прибрано; але семантичні правила, яких
   схема виразити не може (заперечення≠null, дата-запиту≠дата-рішення,
   плейсхолдер≠значення) — залишені і посилені, разом з контрастним
   прикладом (few-shot), розробленим саме під ці правила.

**Два постійні застереження з попередніх прогонів (актуальні й тут):**
1. Enum-обмеження JSON-схеми **не завжди** реально дотримується моделями
   (бачили `rank="Майор"` замість enum-значення `"майор"`, і подібні випадки
   з інших полів) — grammar-constrained decoding в цьому стеку не є 100%
   гарантією, попри теорію. Точні відсотки нижче варто читати з цим
   застереженням.
2. Кейс A4 (частково редагована дата, лише місяць+рік відомі) — усі моделі,
   протестовані досі в цій сесії, вигадували конкретне число замість `null`.
   Очікуємо перевірити, чи повториться це й тут.

**N_REPEATS=2** — з GPU-прискоренням достатньо швидко для повторів, які
підтверджують стабільність (не лише довіряти детермінізму на слово).

### Налаштування Colab (запускати в Colab, потребує GPU-рантайм: Runtime → Change runtime type → GPU)

In [2]:
# Встановлення llama-cpp-python з CUDA - готове (prebuilt) колесо, НЕ компіляція
# з вихідників. CMAKE_ARGS="-DGGML_CUDA=on" (попередня версія цієї клітинки)
# компілює C++/CUDA-код нативно і може "висіти" без видимого прогресу 20-40+
# хвилин у Colab - той самий клас проблеми, який ми вже бачили локально
# (там не було nmake/CMake, тут компіляція просто дуже повільна). Готове
# колесо для CUDA 12.4 (стандартна версія в Colab станом на 2026) ставиться
# за секунди.
# Check GPU details
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

# Enable CUDA support and compile with parallel build flags
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install llama-cpp-python --no-cache-dir

# Install additional required packages
!pip install -q huggingface_hub


name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

MODELS_DIR = Path("/content/models")
MODELS_DIR.mkdir(exist_ok=True)

DOWNLOADS = [
    ("INSAIT-Institute/MamayLM-Gemma-3-12B-IT-v2.0-GGUF", "MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M.gguf"),
    ("lapa-llm/lapa-v0.1.2-instruct-GGUF", "lapa-v0.1.2-instruct-Q4_K_M.gguf"),
]

def model_key(fname: str) -> str:
    # Обрізає лише ".Q4_K_M.gguf"/".gguf" в кінці - fname.split(".")[0] ламається
    # на версіях із крапкою в назві (напр. "Qwen3.6-27B" -> хибно дало б "Qwen3").
    name = fname
    for suffix in (".Q4_K_M.gguf", ".gguf"):
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name

MODEL_PATHS = {}
for repo, fname in DOWNLOADS:
    path = hf_hub_download(repo_id=repo, filename=fname, local_dir=str(MODELS_DIR))
    MODEL_PATHS[model_key(fname)] = Path(path)
    print("downloaded:", path)

MODEL_PATHS


### Спільні утиліти

In [4]:
import re, json, statistics, time, gc
from llama_cpp import Llama

N_REPEATS = 2

RANK_ENUM = [
    "солдат", "старший солдат", "молодший сержант", "сержант", "старший сержант",
    "старшина", "прапорщик", "молодший лейтенант", "лейтенант", "старший лейтенант",
    "капітан", "майор", "підполковник", "полковник", "генерал", "головний сержант",
    None,
]
# НЕ авторитетний список (немає офіційного джерела в контексті проєкту) -
# зведено з трьох різних чернеток цієї сесії. Питання для Анни/військових:
# чи є канонічний перелік звань, на який варто спиратись.

LEAVE_CATEGORY_ENUM = [
    "щорічна основна", "додаткова", "за сімейними обставинами",
    "за станом здоров'я", "без збереження грошового забезпечення", None,
]
# Так само не звірено з офіційним джерелом - ілюстративний словник для тесту.

def normalize_value(v):
    if v is None:
        return None
    if isinstance(v, str):
        v = v.strip()
        if v.lower() in ("", "null", "невідомо", "не вказано", "n/a", "none"):
            return None
        if re.fullmatch(r"-?\d+", v):
            return int(v)
        return v
    return v

def dict_key(d):
    return json.dumps(d, sort_keys=True, ensure_ascii=False) if isinstance(d, dict) else repr(d)


### Задача А — екстракція шумних полів

Кейси A1-A3 - дослівні цитати реального OCR наших власних шаблонних документів
(`docs/відпустка/`), уже розпізнаних Surya в попередніх ноутбуках цієї сесії.
Кейс A4 - реальний (уже редагований у джерелі) OCR-текст довідки ВЛК. Кейс A5 -
**чесно позначений синтетичний** приклад (не з реального сканованого документа) -
тестує пастку "дата рапорту ≠ дата рішення", не потребує реального джерела для
цієї логічної перевірки.

In [5]:
SYSTEM_PROMPT_A = (
    "Ти - модуль екстракції структурованих даних із українських військових "
    "документів для системи, що зберігає результат у типізованих колонках "
    "бази даних PostgreSQL.\n\n"
    "<rules>\n"
    "1. Заповнюй лише ті поля, для яких є пряма textual підстава в документі. "
    "Немає підстави - null.\n"
    "2. Явне заперечення в тексті (напр. \"не надана\", \"не пов'язано\", "
    "\"не надано\") - це підтверджена відповідь (false), а не null. Null - лише "
    "для \"не вказано\", не для \"вказано у формі заперечення\".\n"
    "3. Якщо замість реального значення в тексті стоїть заглушка/плейсхолдер "
    "(\"XXX\", \"_____\", \"[REDACTED]\") чи родова назва поля замість значення "
    "(напр. \"ПІБ\" на місці імені, \"звання\" на місці звання) - це так само "
    "null, не буквальне значення заглушки.\n"
    "4. Дати завжди у форматі DD.MM.YYYY. Якщо дата вказана лише частково "
    "(напр. відомий місяць, але не число) - null, не приблизна дата.\n"
    "5. Прізвище копіюй ТОЧНО як воно записано в тексті (включно з регістром і "
    "відмінком) - нормалізацію робить окремий детермінований крок системи, не "
    "твоя задача.\n"
    "6. Не обчислюй і не вигадуй значення: не виводь дату закінчення з дати "
    "початку і тривалості, якщо вона не вказана прямо; не виводь дату вступу в "
    "силу рішення з дати самого запиту/рапорту - документ-запит ще не є "
    "рішенням, тому дата рішення (якщо окремо не вказана) - null.\n"
    "</rules>\n\n"
    "<example>\n"
    "Текст: \"Прошу надати відрядження терміном на 5 діб з 12.03.2024. "
    "Обов'язки чергового по частині покласти на посадову особу (звання, ПІБ). "
    "Транспорт для виїзду не надано. Рішення про відрядження буде ухвалене "
    "окремим наказом.\"\n"
    "Поля: duration_days, start_date, decision_date, responsible_officer_rank, "
    "responsible_officer_surname, transport_provided\n"
    "Відповідь: {\"duration_days\": 5, \"start_date\": \"12.03.2024\", "
    "\"decision_date\": null, \"responsible_officer_rank\": null, "
    "\"responsible_officer_surname\": null, \"transport_provided\": false}\n"
    "(decision_date - null: 12.03.2024 - дата запиту, а не дата рішення, якого "
    "ще немає. responsible_officer_* - null: \"звання, ПІБ\" - родова назва "
    "поля, не значення. transport_provided - false, не null: \"не надано\" - "
    "це пряма відповідь-заперечення, а не відсутність інформації.)\n"
    "</example>"
)

TASK_A_CASES = [
    {
        "label": "A1: raport-optimized (реальний OCR, XXX-плейсхолдери)",
        "text": (
            "Командиру військової частини XXX\nРАПОРТ\nПрошу вас надати мені "
            "частину щорічної основної відпустки терміном на 10 (десять) діб із "
            "01.10.2023 року.\nВідпустку буду проводити за адресою: країна XXX, "
            "місто XXX, вул. XXX (адреса латиницею). Телефон для оповіщення: "
            "XXX.\n*посада/підрозділ* військової частини XXX.\nxx.xx.2023p.\n"
            "ПІБ"
        ),
        "fields": ["rank", "surname", "leave_category", "leave_days", "leave_start_date", "unit"],
        "schema": {
            "type": "object",
            "properties": {
                "rank": {"type": ["string", "null"], "enum": RANK_ENUM},
                "surname": {"type": ["string", "null"]},
                "leave_category": {"type": ["string", "null"], "enum": LEAVE_CATEGORY_ENUM},
                "leave_days": {"type": ["integer", "null"]},
                "leave_start_date": {"type": ["string", "null"]},
                "unit": {"type": ["string", "null"]},
            },
            "required": ["rank", "surname", "leave_category", "leave_days", "leave_start_date", "unit"],
        },
        "golden": {
            "rank": None, "surname": None,
            "leave_category": "щорічна основна",
            "leave_days": 10, "leave_start_date": "01.10.2023", "unit": None,
        },
    },
    {
        "label": "A2: 1.webp (реальний OCR, повністю заповнений)",
        "text": (
            "Начальнику штабу-заступнику командира військової частини А0000\n"
            "РАПОРТ\nПрошу Вас надати мені частину щорічної основної відпустки "
            "за 2023 рік терміном на 10 діб з 01 січні 2023.\nПомічник "
            "начальника штабу\nМайор                                    "
            "В.ПУПКІН"
        ),
        "fields": ["rank", "surname", "leave_category", "leave_days", "leave_start_date", "unit"],
        "schema": {
            "type": "object",
            "properties": {
                "rank": {"type": ["string", "null"], "enum": RANK_ENUM},
                "surname": {"type": ["string", "null"]},
                "leave_category": {"type": ["string", "null"], "enum": LEAVE_CATEGORY_ENUM},
                "leave_days": {"type": ["integer", "null"]},
                "leave_start_date": {"type": ["string", "null"]},
                "unit": {"type": ["string", "null"]},
            },
            "required": ["rank", "surname", "leave_category", "leave_days", "leave_start_date", "unit"],
        },
        "golden": {
            "rank": "майор", "surname": "В.ПУПКІН",
            "leave_category": "щорічна основна",
            "leave_days": 10, "leave_start_date": "01.01.2023", "unit": "А0000",
        },
    },
    {
        "label": "A3: public (реальний OCR, порожній бланк із фіксованим боілерплейтом)",
        "text": (
            "Командиру В/ч _____\nРапорт\nПрошу надати мені, _____ (звання, "
            "ПІБ), відпустку на _____ діб за сімейними обставинами.\n"
            "Відпустку буду проводити за адресою: _____\nТел.: _____"
        ),
        "fields": ["rank", "surname", "leave_category", "leave_days", "leave_start_date", "unit"],
        "schema": {
            "type": "object",
            "properties": {
                "rank": {"type": ["string", "null"], "enum": RANK_ENUM},
                "surname": {"type": ["string", "null"]},
                "leave_category": {"type": ["string", "null"], "enum": LEAVE_CATEGORY_ENUM},
                "leave_days": {"type": ["integer", "null"]},
                "leave_start_date": {"type": ["string", "null"]},
                "unit": {"type": ["string", "null"]},
            },
            "required": ["rank", "surname", "leave_category", "leave_days", "leave_start_date", "unit"],
        },
        "golden": {
            "rank": None, "surname": None,
            "leave_category": "за сімейними обставинами",  # фіксовано в шаблоні, не бланк
            "leave_days": None, "leave_start_date": None, "unit": None,
        },
    },
    {
        "label": "A4: довідка ВЛК (реальний, уже редагований у джерелі OCR)",
        "text": (
            "ДОВІДКА військово-лікарської комісії\nсолдат\n(військове звання, "
            "прізвання, ім'я та по батькові)\nВійськова частина: А\nв ЗСУ з "
            "[REDACTED] квітня 2022 року призваний [REDACTED] РТЦК та СП\n"
            "Проведено медичний огляд ВЛК КНП [REDACTED] клінічна лікарня "
            "м. Києва. [REDACTED] липня 2023 року.\nДіагноз та постанова ВЛК "
            "про причинний зв'язок захворювання (травми, поранення, контузії, "
            "каліцтва): Стан після мінно - вибухової травми (23.06.2023 р.); "
            "закритої черепно-мозкової травми середнього ступеню важкості; "
            "струсу головного мозку; вестибулярного синдрому; акубаротравми з "
            "пошкодженням обох барабанних перетинок лікованих оперативно: "
            "двобічна мірингопластика (28.06.2023р.).\nЗа наказом МОЗ від "
            "04.07.2007 № 370 травма легка.\nТравма, ТАК, пов'язана з "
            "проходженням військової служби (довідка про обставлення травми не "
            "надана).\nНа підставі статті 81 графи II Розкладу хвороб, "
            "потребує відпустки за станом здоров'я на 30 (тридцять) "
            "календарних днів."
        ),
        "fields": ["rank", "surname", "unit", "exam_date", "injury_date",
                   "injury_connected_to_service", "injury_circumstances_certificate_provided",
                   "sick_leave_days"],
        "schema": {
            "type": "object",
            "properties": {
                "rank": {"type": ["string", "null"], "enum": RANK_ENUM},
                "surname": {"type": ["string", "null"]},
                "unit": {"type": ["string", "null"]},
                "exam_date": {"type": ["string", "null"]},
                "injury_date": {"type": ["string", "null"]},
                "injury_connected_to_service": {"type": ["boolean", "null"]},
                "injury_circumstances_certificate_provided": {"type": ["boolean", "null"]},
                "sick_leave_days": {"type": ["integer", "null"]},
            },
            "required": ["rank", "surname", "unit", "exam_date", "injury_date",
                         "injury_connected_to_service", "injury_circumstances_certificate_provided",
                         "sick_leave_days"],
        },
        "golden": {
            "rank": "солдат",
            "surname": None,  # лише родовий підпис-мітка поля, ім'я не вказано
            "unit": None,     # "Військова частина: А" - неповне значення, не повний код
            "exam_date": None,  # день редаговано, лише місяць+рік відомі - неповна дата
            "injury_date": "23.06.2023",  # повна дата всередині діагнозу, не редагована
            "injury_connected_to_service": True,
            "injury_circumstances_certificate_provided": False,
            "sick_leave_days": 30,
        },
    },
    {
        "label": "A5: рапорт на звільнення (СИНТЕТИЧНИЙ приклад, не з реального документа - тестує пастку request!=decision)",
        "text": (
            "Командиру військової частини B5678\nРАПОРТ\nПрошу вашого "
            "сприяння щодо звільнення мене з військової служби у запас "
            "Збройних Сил України на підставі п.2 ч.4 ст.26 Закону України "
            "«Про військовий обов'язок і військову службу» у зв'язку з "
            "вихованням дитини до 18 років без матері.\nГоловний сержант\n"
            "21.11.2023                                    Сидоренко О.П."
        ),
        "fields": ["rank", "surname", "unit", "discharge_legal_ground",
                   "report_date", "discharge_effective_date"],
        "schema": {
            "type": "object",
            "properties": {
                "rank": {"type": ["string", "null"], "enum": RANK_ENUM},
                "surname": {"type": ["string", "null"]},
                "unit": {"type": ["string", "null"]},
                "discharge_legal_ground": {"type": ["string", "null"]},
                "report_date": {"type": ["string", "null"]},
                "discharge_effective_date": {"type": ["string", "null"]},
            },
            "required": ["rank", "surname", "unit", "discharge_legal_ground",
                         "report_date", "discharge_effective_date"],
        },
        "golden": {
            "rank": "головний сержант", "surname": "Сидоренко О.П.", "unit": "B5678",
            "discharge_legal_ground": "п.2 ч.4 ст.26 Закону України «Про військовий обов'язок і військову службу», виховання дитини до 18 років без матері",
            "report_date": "21.11.2023",
            "discharge_effective_date": None,  # пастка: 21.11.2023 - дата рапорту, не дата рішення
        },
    },
]

def build_task_a_messages(case):
    fields_str = ", ".join(case["fields"])
    user = f"<document>\n{case['text']}\n</document>\n\nВитягни поля: {fields_str}."
    return [
        {"role": "system", "content": SYSTEM_PROMPT_A},
        {"role": "user", "content": user},
    ]

def score_task_a(golden, predicted):
    if not isinstance(predicted, dict):
        return {"valid_json": False, "field_scores": {}, "accuracy": 0.0}
    field_scores = {}
    for field, gold_val in golden.items():
        pred_val = normalize_value(predicted.get(field))
        gold_norm = normalize_value(gold_val)
        field_scores[field] = (pred_val == gold_norm)
    accuracy = sum(field_scores.values()) / len(field_scores)
    return {"valid_json": True, "field_scores": field_scores, "accuracy": accuracy}


### Задача Б — резолюція терміну в довідник вимір+значення

**Застереження:** цей словник вимір+значення - ілюстративний для методики тесту,
НЕ остаточна схема бази (open-questions.md прямо каже, що ця схема ще не
узгоджена з військовими).

In [6]:
DIMENSION_VALUE_VOCAB = """
- вимір=vacation, значення=active - особа перебуває у відпустці ("у відпустці", "у щорічній відпустці" тощо)
- вимір=discharge, значення=pending - подано рапорт на звільнення, рішення ще не ухвалене
- вимір=discharge, значення=discharged - особа вже звільнена з військової служби
- вимір=medical_treatment, значення=hospitalized - особа проходить лікування у шпиталі
- вимір=duty_station, значення=deployed - особа у відрядженні
- вимір=duty_station, значення=at_permanent_post - особа на постійному місці служби
""".strip()

SYSTEM_PROMPT_B = (
    "Ти визначаєш, якому виміру й значенню з контрольованого довідника "
    "відповідає термін із документа.\n\n"
    f"<vocabulary>\n{DIMENSION_VALUE_VOCAB}\n</vocabulary>\n\n"
    "<example>\n"
    "Термін: \"на лікарняному\"\n"
    "Відповідь: {\"recognized\": false, \"dimension\": null, \"value\": null}\n"
    "(Це семантично близько до \"проходить лікування у шпиталі\", але не той "
    "самий термін - точного відповідника в довіднику немає, тому не підбирай "
    "найближчий, постав recognized=false.)\n"
    "</example>\n\n"
    "Якщо термін НЕ відповідає ТОЧНО жодному з варіантів довідника - постав "
    "recognized=false, dimension=null, value=null. Не підбирай найближчий за "
    "змістом варіант."
)

TASK_B_SCHEMA = {
    "type": "object",
    "properties": {
        "recognized": {"type": "boolean"},
        "dimension": {"type": ["string", "null"],
                      "enum": ["vacation", "discharge", "medical_treatment", "duty_station", None]},
        "value": {"type": ["string", "null"],
                  "enum": ["active", "pending", "discharged", "hospitalized", "deployed", "at_permanent_post", None]},
    },
    "required": ["recognized", "dimension", "value"],
}

TASK_B_CASES = [
    {"term": "у відпустці", "golden": {"recognized": True, "dimension": "vacation", "value": "active"}},
    {"term": "перебуває у щорічній відпустці", "golden": {"recognized": True, "dimension": "vacation", "value": "active"}},
    {"term": "подав рапорт на звільнення", "golden": {"recognized": True, "dimension": "discharge", "value": "pending"}},
    {"term": "вже звільнений із служби", "golden": {"recognized": True, "dimension": "discharge", "value": "discharged"}},
    {"term": "перебуває на лікуванні у шпиталі", "golden": {"recognized": True, "dimension": "medical_treatment", "value": "hospitalized"}},
    {"term": "у відрядженні", "golden": {"recognized": True, "dimension": "duty_station", "value": "deployed"}},
    {"term": "на постійному місці служби, в частині", "golden": {"recognized": True, "dimension": "duty_station", "value": "at_permanent_post"}},
    {"term": "у полоні", "golden": {"recognized": False, "dimension": None, "value": None}},
]

def build_task_b_messages(term):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_B},
        {"role": "user", "content": f"<term>{term}</term>"},
    ]

def score_task_b(golden, predicted):
    if not isinstance(predicted, dict):
        return False
    return all(predicted.get(k) == v for k, v in golden.items())


### Задача В — вибір шаблону запиту + параметри

**unit_number** - окремий, вужчий атрибут від коду військової частини
("А0000" тощо з Задачі А) - номер підрозділу ВСЕРЕДИНІ частини (напр. номер
батальйону). Це дві різні речі, які легко переплутати - явно розділено в
промпті нижче.

In [7]:
QUERY_TEMPLATES = """
- COUNT_BY_STATUS(dimension, value) - порахувати людей за виміром+значенням
- LIST_BY_STATUS(dimension, value) - перелічити людей за виміром+значенням
- COUNT_BY_STATUS_DATE_RANGE(dimension, value, date_from, date_to) - порахувати за період
- COUNT_BY_STATUS_UNIT(dimension, value, unit_number) - порахувати у конкретному підрозділі
- UNSUPPORTED - жоден шаблон не підходить (включно з запитами поза темою і запитами-порадами/думками, а не підрахунком/списком)
""".strip()

SYSTEM_PROMPT_C = (
    "Ти обираєш шаблон запиту до бази даних із фіксованого списку і заповнюєш "
    "параметри. Ти ніколи не пишеш SQL сам.\n\n"
    f"<templates>\n{QUERY_TEMPLATES}\n</templates>\n\n"
    f"<vocabulary>\n{DIMENSION_VALUE_VOCAB}\n</vocabulary>\n\n"
    "unit_number - це ЧИСЛО (напр. \"3 батальйон\" -> unit_number=3) - окремий, "
    "вужчий атрибут від коду військової частини (\"А0000\" тощо) - номер "
    "підрозділу всередині частини, не код самої частини. Дати - у форматі "
    "DD.MM.YYYY.\n\n"
    "<example>\n"
    "Питання: \"Скільки людей у відрядженні в 7 роті?\"\n"
    "Відповідь: {\"template\": \"COUNT_BY_STATUS_UNIT\", \"params\": "
    "{\"dimension\": \"duty_station\", \"value\": \"deployed\", "
    "\"unit_number\": 7, \"date_from\": null, \"date_to\": null}}\n"
    "</example>\n\n"
    "Якщо жоден шаблон не підходить - template=UNSUPPORTED, усі параметри null."
)

TASK_C_SCHEMA = {
    "type": "object",
    "properties": {
        "template": {
            "type": "string",
            "enum": ["COUNT_BY_STATUS", "LIST_BY_STATUS", "COUNT_BY_STATUS_DATE_RANGE",
                     "COUNT_BY_STATUS_UNIT", "UNSUPPORTED"],
        },
        "params": {
            "type": "object",
            "properties": {
                "dimension": {"type": ["string", "null"],
                              "enum": ["vacation", "discharge", "medical_treatment", "duty_station", None]},
                "value": {"type": ["string", "null"],
                          "enum": ["active", "pending", "discharged", "hospitalized", "deployed", "at_permanent_post", None]},
                "unit_number": {"type": ["integer", "null"]},
                "date_from": {"type": ["string", "null"]},
                "date_to": {"type": ["string", "null"]},
            },
            "required": ["dimension", "value", "unit_number", "date_from", "date_to"],
        },
    },
    "required": ["template", "params"],
}

TASK_C_CASES = [
    {
        "question": "Скільки людей зараз перебувають у відпустці?",
        "golden": {"template": "COUNT_BY_STATUS",
                   "params": {"dimension": "vacation", "value": "active", "unit_number": None, "date_from": None, "date_to": None}},
    },
    {
        "question": "Хто зараз перебуває у відпустці?",
        "golden": {"template": "LIST_BY_STATUS",
                   "params": {"dimension": "vacation", "value": "active", "unit_number": None, "date_from": None, "date_to": None}},
    },
    {
        "question": "Скільки людей пішли у відпустку з 01.06.2026 по 30.06.2026?",
        "golden": {"template": "COUNT_BY_STATUS_DATE_RANGE",
                   "params": {"dimension": "vacation", "value": "active", "unit_number": None,
                              "date_from": "01.06.2026", "date_to": "30.06.2026"}},
    },
    {
        "question": "Скільки людей у 3 батальйоні зараз перебувають на лікуванні?",
        "golden": {"template": "COUNT_BY_STATUS_UNIT",
                   "params": {"dimension": "medical_treatment", "value": "hospitalized", "unit_number": 3,
                              "date_from": None, "date_to": None}},
    },
    {
        "question": "Яка сьогодні погода у Києві?",
        "golden": {"template": "UNSUPPORTED",
                   "params": {"dimension": None, "value": None, "unit_number": None, "date_from": None, "date_to": None}},
    },
    {
        "question": "Порадьте, кого краще відправити у відрядження наступного місяця.",
        "golden": {"template": "UNSUPPORTED",
                   "params": {"dimension": None, "value": None, "unit_number": None, "date_from": None, "date_to": None}},
    },
]

def build_task_c_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_C},
        {"role": "user", "content": f"<question>{question}</question>"},
    ]

def score_task_c(golden, predicted):
    if not isinstance(predicted, dict):
        return False
    if predicted.get("template") != golden["template"]:
        return False
    pred_params = {k: normalize_value(v) for k, v in (predicted.get("params") or {}).items()}
    gold_params = {k: normalize_value(v) for k, v in golden["params"].items()}
    return pred_params == gold_params


### Прогін: одна модель повністю (GPU, усі 3 задачі), вивантаження, наступна

In [8]:
def load_model_resilient(model_path):
    """
    Спроба повного offload на GPU (швидко); якщо модель не влазить у VRAM
    (напр. Qwen3.6-27B, 16.8 GB, впритул/більше за 16 GB VRAM типової T4 на
    Colab) - відкат на частковий/CPU offload, щоб один завеликий кандидат не
    зупиняв увесь прогін, лише сповільнював саме цю модель.
    """
    try:
        return Llama(model_path=str(model_path), n_ctx=4096, n_gpu_layers=-1, verbose=False), "GPU (повний offload)"
    except Exception as e:
        print(f"  Повний GPU-offload не вдався ({type(e).__name__}: {e}) - пробую частковий offload (20 шарів)...")
        try:
            return Llama(model_path=str(model_path), n_ctx=4096, n_gpu_layers=20, verbose=False), "частковий GPU-offload (20 шарів)"
        except Exception as e2:
            print(f"  Частковий offload теж не вдався ({type(e2).__name__}: {e2}) - відкат на CPU-only...")
            return Llama(model_path=str(model_path), n_ctx=4096, n_gpu_layers=0, verbose=False), "CPU-only (повільно)"

def run_all_tasks_for_model(model_name: str, model_path) -> dict:
    print(f"=== Завантаження {model_name} ({model_path.stat().st_size / 1e9:.1f} GB) ===")
    t0 = time.time()
    llm, load_mode = load_model_resilient(model_path)
    print(f"  завантажено за {time.time() - t0:.0f}с ({load_mode})")

    def call(messages, schema):
        t0 = time.time()
        resp = llm.create_chat_completion(
            messages=messages, temperature=0,
            response_format={"type": "json_object", "schema": schema},
        )
        elapsed = time.time() - t0
        print(f"    відповідь за {elapsed:.1f}с")
        try:
            parsed = json.loads(resp["choices"][0]["message"]["content"])
        except json.JSONDecodeError:
            parsed = None
        return parsed, elapsed

    task_a = []
    for case in TASK_A_CASES:
        raw = [call(build_task_a_messages(case), case["schema"]) for _ in range(N_REPEATS)]
        parsed_list = [r[0] for r in raw]
        times_list = [r[1] for r in raw]
        scores = [score_task_a(case["golden"], p) for p in parsed_list]
        accuracies = [s["accuracy"] for s in scores]
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        result = {
            "case": case["label"], "mean_accuracy": statistics.mean(accuracies),
            "min_accuracy": min(accuracies), "max_accuracy": max(accuracies),
            "consistent_across_repeats": consistent, "parsed_list": parsed_list,
            "field_scores_list": [s["field_scores"] for s in scores],
            "times_s": times_list, "mean_time_s": statistics.mean(times_list),
        }
        task_a.append(result)
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] Задача А / {case['label']}: точність={result['mean_accuracy']:.0%}, "
              f"час={result['mean_time_s']:.1f}с — {stability}")

    task_b = []
    for case in TASK_B_CASES:
        raw = [call(build_task_b_messages(case["term"]), TASK_B_SCHEMA) for _ in range(N_REPEATS)]
        parsed_list = [r[0] for r in raw]
        times_list = [r[1] for r in raw]
        correct_list = [score_task_b(case["golden"], p) for p in parsed_list]
        correct_rate = sum(correct_list) / len(correct_list)
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        task_b.append({
            "term": case["term"], "golden": case["golden"], "parsed_list": parsed_list,
            "correct_rate": correct_rate, "consistent_across_repeats": consistent,
            "times_s": times_list, "mean_time_s": statistics.mean(times_list),
        })
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] Задача Б / \"{case['term']}\" -> {parsed_list} "
              f"(час={statistics.mean(times_list):.1f}с) — {stability}")

    task_c = []
    for case in TASK_C_CASES:
        raw = [call(build_task_c_messages(case["question"]), TASK_C_SCHEMA) for _ in range(N_REPEATS)]
        parsed_list = [r[0] for r in raw]
        times_list = [r[1] for r in raw]
        correct_list = [score_task_c(case["golden"], p) for p in parsed_list]
        correct_rate = sum(correct_list) / len(correct_list)
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        task_c.append({
            "question": case["question"], "golden": case["golden"], "parsed_list": parsed_list,
            "correct_rate": correct_rate, "consistent_across_repeats": consistent,
            "times_s": times_list, "mean_time_s": statistics.mean(times_list),
        })
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] Задача В / \"{case['question']}\" -> {parsed_list} "
              f"(час={statistics.mean(times_list):.1f}с) — {stability}")

    del llm
    gc.collect()
    print(f"=== {model_name} вивантажено ===\n")
    return {"task_a": task_a, "task_b": task_b, "task_c": task_c, "load_mode": load_mode}

RESULTS = {}
for model_name, model_path in MODEL_PATHS.items():
    try:
        RESULTS[model_name] = run_all_tasks_for_model(model_name, model_path)
    except Exception as e:
        print(f"!!! {model_name} провалився повністю ({type(e).__name__}: {e}) - пропускаю, інші моделі тестуватимуться далі\n")
        RESULTS[model_name] = None


=== Завантаження MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M (7.3 GB) ===


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


  завантажено за 33с (GPU (повний offload))
    відповідь за 7.8с
    відповідь за 6.9с
[MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M] Задача А / A1: raport-optimized (реальний OCR, XXX-плейсхолдери): точність=50%, час=7.4с — стабільно
    відповідь за 6.8с
    відповідь за 7.3с
[MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M] Задача А / A2: 1.webp (реальний OCR, повністю заповнений): точність=50%, час=7.1с — стабільно
    відповідь за 4.1с
    відповідь за 4.8с
[MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M] Задача А / A3: public (реальний OCR, порожній бланк із фіксованим боілерплейтом): точність=100%, час=4.5с — стабільно
    відповідь за 8.6с
    відповідь за 8.8с
[MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M] Задача А / A4: довідка ВЛК (реальний, уже редагований у джерелі OCR): точність=75%, час=8.7с — стабільно
    відповідь за 13.4с
    відповідь за 13.0с
[MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M] Задача А / A5: рапорт на звільнення (СИНТЕТИЧНИЙ приклад, не з реального документа - тестує пастку request!=decision): точність=83%

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


  завантажено за 33с (GPU (повний offload))
    відповідь за 7.4с
    відповідь за 6.8с
[gemma-3-12b-it-Q4_K_M] Задача А / A1: raport-optimized (реальний OCR, XXX-плейсхолдери): точність=67%, час=7.1с — стабільно
    відповідь за 7.3с
    відповідь за 7.8с
[gemma-3-12b-it-Q4_K_M] Задача А / A2: 1.webp (реальний OCR, повністю заповнений): точність=67%, час=7.5с — стабільно
    відповідь за 4.3с
    відповідь за 4.9с
[gemma-3-12b-it-Q4_K_M] Задача А / A3: public (реальний OCR, порожній бланк із фіксованим боілерплейтом): точність=100%, час=4.6с — стабільно
    відповідь за 8.5с
    відповідь за 8.7с
[gemma-3-12b-it-Q4_K_M] Задача А / A4: довідка ВЛК (реальний, уже редагований у джерелі OCR): точність=62%, час=8.6с — стабільно
    відповідь за 10.9с
    відповідь за 10.6с
[gemma-3-12b-it-Q4_K_M] Задача А / A5: рапорт на звільнення (СИНТЕТИЧНИЙ приклад, не з реального документа - тестує пастку request!=decision): точність=50%, час=10.8с — стабільно
    відповідь за 2.3с
    відповідь за 1.

### Підсумок

In [9]:
print("=" * 70)
print(f"ПІДСУМОК ({len(MODEL_PATHS)} моделей, N={N_REPEATS} повтори)")
print("=" * 70)
for model_name, r in RESULTS.items():
    if r is None:
        print(f"\n{model_name}: ПРОВАЛ - модель не вдалось завантажити/протестувати взагалі")
        continue
    a_acc = statistics.mean(x["mean_accuracy"] for x in r["task_a"])
    a_consistent = sum(x["consistent_across_repeats"] for x in r["task_a"])
    b_acc = statistics.mean(x["correct_rate"] for x in r["task_b"])
    b_consistent = sum(x["consistent_across_repeats"] for x in r["task_b"])
    c_acc = statistics.mean(x["correct_rate"] for x in r["task_c"])
    c_consistent = sum(x["consistent_across_repeats"] for x in r["task_c"])
    all_times = ([t for x in r["task_a"] for t in x["times_s"]]
                 + [t for x in r["task_b"] for t in x["times_s"]]
                 + [t for x in r["task_c"] for t in x["times_s"]])
    print(f"\n{model_name} (завантаження: {r.get('load_mode', '?')}):")
    print(f"  Задача А: точність={a_acc:.0%}, стабільно на {a_consistent}/{len(r['task_a'])} кейсів, "
          f"середній час={statistics.mean(x['mean_time_s'] for x in r['task_a']):.1f}с")
    print(f"  Задача Б: точність={b_acc:.0%}, стабільно на {b_consistent}/{len(r['task_b'])} кейсів, "
          f"середній час={statistics.mean(x['mean_time_s'] for x in r['task_b']):.1f}с")
    print(f"  Задача В: точність={c_acc:.0%}, стабільно на {c_consistent}/{len(r['task_c'])} кейсів, "
          f"середній час={statistics.mean(x['mean_time_s'] for x in r['task_c']):.1f}с")
    print(f"  Загалом: середній час відповіді={statistics.mean(all_times):.1f}с "
          f"(мін={min(all_times):.1f}с, макс={max(all_times):.1f}с, n={len(all_times)} викликів)")

print()
for model_name, r in RESULTS.items():
    if r is None:
        continue
    for res in r["task_a"]:
        wrong = {f: rate for f, rate in
                 {fn: sum(1 for fs in res["field_scores_list"] if not fs.get(fn, False)) / len(res["field_scores_list"])
                  for fn in (res["field_scores_list"][0].keys() if res["field_scores_list"] else [])}.items()
                 if rate > 0}
        if wrong:
            print(f"[{model_name}] {res['case']}: помилки по полях {wrong}")
            print(f"  приклад: {res['parsed_list'][0]}")


ПІДСУМОК (3 моделей, N=2 повтори)

MamayLM-Gemma-3-12B-IT-v2.0-Q4_K_M (завантаження: GPU (повний offload)):
  Задача А: точність=72%, стабільно на 5/5 кейсів, середній час=8.2с
  Задача Б: точність=100%, стабільно на 8/8 кейсів, середній час=2.0с
  Задача В: точність=100%, стабільно на 6/6 кейсів, середній час=4.7с
  Загалом: середній час відповіді=4.5с (мін=1.3с, макс=13.4с, n=38 викликів)

gemma-3-12b-it-Q4_K_M (завантаження: GPU (повний offload)):
  Задача А: точність=69%, стабільно на 5/5 кейсів, середній час=7.7с
  Задача Б: точність=88%, стабільно на 8/8 кейсів, середній час=1.9с
  Задача В: точність=100%, стабільно на 6/6 кейсів, середній час=4.8с
  Загалом: середній час відповіді=4.3с (мін=1.3с, макс=10.9с, n=38 викликів)

Qwen3-8B-Q4_K_M (завантаження: GPU (повний offload)):
  Задача А: точність=66%, стабільно на 5/5 кейсів, середній час=4.9с
  Задача Б: точність=100%, стабільно на 8/8 кейсів, середній час=1.2с
  Задача В: точність=100%, стабільно на 6/6 кейсів, середній час=2